# Kinetic action and straight geodesics

This notebook generates `fig:dynamic-action-kinetic-energy`.  For a single characteristic path $\gamma:[0,1]\to\mathbb R^2$, the dynamic formulation uses the kinetic action
$$
\mathcal A(\gamma)=\int_0^1 \lVert \dot\gamma(t)\rVert^2\,dt.
$$
Among all absolutely continuous paths with fixed endpoints, Jensen's inequality selects the constant-speed straight segment.  The figure compares this minimizer with a curved detour and encodes the local kinetic energy $\lVert \dot\gamma(t)\rVert^2$ by line thickness and small bars.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    DIRAC_MARKER_SIZE,
    interp_color,
    remove_axes,
    save_pdf,
    setup_matplotlib,
    figure_dir,
)

setup_matplotlib()
rng = np.random.default_rng(2027)

from matplotlib.collections import LineCollection


## Two paths with the same endpoints

The straight path is
$$
\gamma_s(t)=(1-t)x_0+t x_1,
$$
while the curved path adds a sinusoidal detour in the normal direction:
$$
\gamma_c(t)=\gamma_s(t)+a\sin(\pi t)n.
$$
Both have the same endpoints, but the curved path has extra velocity orthogonal to the displacement and therefore larger action.

In [ ]:
x0 = np.array([-1.20, -0.42])
x1 = np.array([1.20, 0.42])
delta = x1 - x0
normal = np.array([-delta[1], delta[0]])
normal = normal / np.linalg.norm(normal)
amplitude = 0.86


def gamma_straight(t):
    t = np.asarray(t)
    return (1 - t)[..., None] * x0 + t[..., None] * x1


def dgamma_straight(t):
    t = np.asarray(t)
    return np.broadcast_to(delta, t.shape + (2,))


def gamma_curved(t):
    t = np.asarray(t)
    return gamma_straight(t) + amplitude * np.sin(np.pi * t)[..., None] * normal


def dgamma_curved(t):
    t = np.asarray(t)
    return dgamma_straight(t) + amplitude * np.pi * np.cos(np.pi * t)[..., None] * normal


def action(vfun, n=2000):
    tt = np.linspace(0.0, 1.0, n)
    speed2 = np.sum(vfun(tt) ** 2, axis=1)
    return float(np.trapezoid(speed2, tt))

A_straight = action(dgamma_straight)
A_curved = action(dgamma_curved)


## Exported panels

The two PDFs are title-free.  Panel labels and numerical action values are added in LaTeX, while the visual encodes time by the red-to-blue color scale.

In [ ]:
fig_name = "dynamic-action-kinetic-energy"
out = figure_dir(fig_name)


def draw_action_panel(path_fun, velocity_fun, filename):
    tt = np.linspace(0.0, 1.0, 180)
    pts = path_fun(tt)
    vel = velocity_fun(tt)
    speed2 = np.sum(vel**2, axis=1)
    speed2_mid = 0.5 * (speed2[:-1] + speed2[1:])
    scaled = speed2_mid / speed2_mid.max()
    widths = 0.65 + 2.15 * scaled

    fig, ax = plt.subplots(figsize=(2.45, 2.1))
    segments = [[pts[k], pts[k + 1]] for k in range(len(tt) - 1)]
    colors = [(*interp_color(float(0.5 * (tt[k] + tt[k + 1]))), 0.88) for k in range(len(tt) - 1)]
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=widths, capstyle="round", zorder=2))
    ax.scatter([x0[0]], [x0[1]], s=1.15 * DIRAC_MARKER_SIZE, color=RED, edgecolor="none", linewidth=0, zorder=4)
    ax.scatter([x1[0]], [x1[1]], s=1.15 * DIRAC_MARKER_SIZE, color=BLUE, edgecolor="none", linewidth=0, zorder=4)

    # Small local-energy bars, kept inside the panel and without axis labels.
    bar_t = np.linspace(0.08, 0.92, 9)
    bar_speed = np.sum(velocity_fun(bar_t) ** 2, axis=1)
    bar_height = 0.08 + 0.28 * bar_speed / max(bar_speed.max(), 1e-12)
    bar_x = np.linspace(-1.05, 1.05, len(bar_t))
    bar_y0 = -1.06
    for bx, h, bt in zip(bar_x, bar_height, bar_t):
        ax.plot([bx, bx], [bar_y0, bar_y0 + h], color=interp_color(float(bt)), lw=2.1, solid_capstyle="round", zorder=3)

    ax.set_xlim(-1.55, 1.55)
    ax.set_ylim(-1.18, 1.18)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.045)
    plt.close(fig)


draw_action_panel(gamma_straight, dgamma_straight, "straight.pdf")
draw_action_panel(gamma_curved, dgamma_curved, "curved.pdf")
